# Denoising Autoencoder on CIFAR-10

**Goal:** Train a convolutional autoencoder to remove Gaussian noise from colour images drawn from the CIFAR-10 dataset.

A *denoising autoencoder* learns to map a corrupted image back to its clean counterpart. By forcing the reconstruction through a low-dimensional bottleneck the encoder must discover compact, noise-invariant representations—making these models useful for tasks ranging from image restoration to unsupervised feature learning.

---

**Contents**
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Noise Augmentation](#2-data-loading--noise-augmentation)
3. [Model Architecture](#3-model-architecture)
4. [Training](#4-training)
5. [Evaluation & Worst-Case Analysis](#5-evaluation--worst-case-analysis)
6. [Hyperparameter Study](#6-hyperparameter-study)
7. [Discussion](#7-discussion)

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# Project modules (model.py / data.py / train.py / evaluate.py must be in the same directory)
from model    import DenoisingAutoencoder
from data     import load_cifar10
from train    import train
from evaluate import evaluate, show_noisy_pairs, show_worst, hyperparameter_study

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 240167723
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Data Loading & Noise Augmentation

CIFAR-10 contains 60,000 32×32 RGB images across 10 classes (50,000 train / 10,000 test).

Noise is applied *on-the-fly* via a torchvision `Lambda` transform that adds zero-mean Gaussian noise and clips the result back to [0, 1]:

$$x_{\text{noisy}} = \text{clip}\bigl(x + \sigma \cdot \epsilon,\; 0,\; 1\bigr), \quad \epsilon \sim \mathcal{N}(0, I)$$

We use $\sigma = 0.2$. Both clean and noisy loaders are built with `shuffle=False` so that parallel iteration with `zip()` always yields correctly matched pairs.

In [ ]:
NOISE_SCALE = 0.2
BATCH_SIZE  = 64

original_trainloader, original_testloader, noisy_trainloader, noisy_testloader = \
    load_cifar10(noise_scale=NOISE_SCALE, batch_size=BATCH_SIZE)

# Sanity check: confirm dataset sizes
n_train = len(original_trainloader.dataset)
n_test  = len(original_testloader.dataset)
print(f'Training images : {n_train:,}')
print(f'Test images     : {n_test:,}')
print(f'Noise scale (σ) : {NOISE_SCALE}')

In [ ]:
# Visualise 10 original / noisy image pairs
show_noisy_pairs(original_testloader, noisy_testloader, num_pairs=10)

## 3. Model Architecture

The autoencoder uses strided convolutions (encoder) and transposed convolutions (decoder) to learn a 512-dimensional bottleneck representation.

```
Input  (3, 32, 32)
  │
  ▼  Conv2d(3→64,   k=4, s=2)  → (64,  16, 16)
  ▼  Conv2d(64→128, k=4, s=2)  → (128,  8,  8)
  ▼  Conv2d(128→256,k=4, s=2)  → (256,  4,  4)
  ▼  Flatten → Linear(4096→512)  ← bottleneck
  ▼  Linear(512→4096) → Unflatten
  ▼  ConvTranspose2d(256→128)   → (128,  8,  8)
  ▼  ConvTranspose2d(128→64)    → (64,  16, 16)
  ▼  ConvTranspose2d(64→3)      → (3,   32, 32)
Output (3, 32, 32)  [Sigmoid → [0,1]]
```

In [ ]:
model = DenoisingAutoencoder()
print(model)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

## 4. Training

- **Loss:** Mean Squared Error (MSE) between reconstructed output and clean target image
- **Optimiser:** Adam with lr = 0.001
- **Epochs:** 10

In [ ]:
NUM_EPOCHS = 10
LR         = 0.001

epoch_losses = train(
    model=model,
    noisy_trainloader=noisy_trainloader,
    original_trainloader=original_trainloader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    device=DEVICE,
    save_path='autoencoder.pth',
)

In [ ]:
# Plot training loss curve
plt.figure(figsize=(7, 4))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_losses, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Mean Training MSE')
plt.title('Training Loss Curve')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=120)
plt.show()

## 5. Evaluation & Worst-Case Analysis

We feed every noisy test image through the trained model and compute the per-image MSE against its clean counterpart. The 20 images with the highest reconstruction error are displayed as triples: **(Original | Noisy | Reconstructed)**.

In [ ]:
results = evaluate(model, noisy_testloader, original_testloader, DEVICE)

print(f'Mean test MSE   : {np.mean(results["errors"]):.6f}')
print(f'Median test MSE : {np.median(results["errors"]):.6f}')
print(f'Max test MSE    : {np.max(results["errors"]):.6f}')

In [ ]:
show_worst(results, num_worst=20)

## 6. Hyperparameter Study

We independently sweep two hyperparameters while holding all others fixed.

| Hyperparameter | Values tested                       | Fixed counterpart |
|----------------|-------------------------------------|-------------------|
| Learning rate  | 0.0001, 0.001, 0.005, 0.01, 0.05   | batch size = 64   |
| Batch size     | 16, 32, 64, 128, 256               | lr = 0.001        |

In [ ]:
hp_results = hyperparameter_study(
    num_epochs=10,
    noise_scale=NOISE_SCALE,
    device=DEVICE,
)

## 7. Discussion

### 7.1 Learning Rate

Within the narrow range originally tested (0.001–0.01), reconstruction error is nearly flat—suggesting the model is not highly sensitive to small learning-rate variations in that regime. However, this stability should not be extrapolated:

- **Too low (e.g. 0.0001):** Adam accumulates slow gradient updates; 10 epochs are insufficient to converge, so test MSE remains elevated.
- **Too high (e.g. 0.05):** Step sizes overshoot the loss minimum, causing oscillation or divergence and producing visibly blurred / artefact-heavy reconstructions.

This U-shaped relationship—high error at both extremes, low error in the middle—is the canonical *learning-rate sensitivity curve* and is confirmed by the wider sweep in Section 6.

### 7.2 Batch Size

Smaller batch sizes (16–32) tend to yield marginally lower reconstruction error. This is consistent with the *large-batch generalisation gap* (Keskar et al., 2017): small-batch stochastic gradients explore the loss landscape more broadly and settle in flatter minima that generalise better, while large batches converge to sharper minima with slightly worse test performance.

At very small batch sizes the benefit diminishes because noisy gradient estimates destabilise training, and wall-clock time per epoch increases substantially due to more frequent parameter updates.

### 7.3 Qualitative Observations

- The worst-reconstructed images (Section 5) tend to have fine textures or high-frequency detail (e.g. animals with complex fur patterns), where the bottleneck cannot perfectly recover all spatial information after compression.
- Reconstructed images are visibly sharper than the noisy inputs, confirming that the autoencoder has learned to suppress noise rather than merely memorise training examples.
- The Sigmoid output activation ensures pixel values remain in [0, 1] without explicit clipping, which avoids gradient saturation at the extremes compared to an unactivated linear output.